# Experiment 3 — Reader-visible fit diagnostics

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** For each policy, fit the Stage-2 selected linear model with the v7
logit-scale training, then show reader-facing diagnostic plots:

- pred-vs-actual scatter with the 45° line
- residual histogram
- calibration-by-decile plot

These are the figures that anchor the paper's "linear is really good" narrative.

In [ ]:
def fit_selected_and_predict(policy):
    feat, model_name, alpha = SELECTED[policy]
    meta, X, y, cols = nu.build_design_matrix(turns, policy, feat)
    pred_df, _ = nu.grouped_cv_predict(
        meta, X, y, model_name,
        n_splits=nu.OUTER_FOLDS,
        fixed_alpha=float(alpha) if (model_name == 'Ridge' and np.isfinite(alpha)) else None)
    return pred_df, nu.cv_metrics_full(pred_df), feat, model_name

results = {p: fit_selected_and_predict(p) for p in POLICIES}
for p in POLICIES:
    _, m, feat, mdl = results[p]
    print(f'{p:<6s}  {feat} / {mdl:<6s}  MAE={m["mae_norm"]:.4f}  R2={m["r2"]:.3f}  bias={m["bias"]:+.4f}')

### Pred-vs-actual + residual histograms

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i, p in enumerate(POLICIES):
    pred_df, m, feat, mdl = results[p]
    nu.pred_vs_actual_plot(pred_df['y_true'], pred_df['y_pred'],
                           title=f'{p}\n{feat} / {mdl}\nMAE={m["mae_norm"]:.3f}, R2={m["r2"]:.3f}',
                           ax=axes[0, i])
    nu.residual_hist(pred_df['y_true'], pred_df['y_pred'],
                     title=f'bias = {m["bias"]:+.3f}', ax=axes[1, i])
plt.tight_layout()
plt.savefig('figE3_diagnostics.png', dpi=140, bbox_inches='tight')
plt.show()

### Decile calibration

Bin observed $\tilde S_t$ into deciles, then plot mean predicted vs mean observed
per bin.  A well-calibrated model follows the 45° line.

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for i, p in enumerate(POLICIES):
    pred_df, _, _, _ = results[p]
    y  = pred_df['y_true'].to_numpy()
    yh = pred_df['y_pred'].to_numpy()
    bins = pd.qcut(y, q=10, duplicates='drop')
    g = pd.DataFrame({'y': y, 'yh': yh, 'bin': bins}).groupby('bin', observed=True)
    ax = axes[i]
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.errorbar(g['y'].mean(), g['yh'].mean(),
                yerr=g['yh'].std(), fmt='o-', lw=1.2, capsize=3, ms=5)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(p); ax.set_xlabel('obs mean (decile)')
    if i == 0:
        ax.set_ylabel('pred mean')
plt.tight_layout()
plt.savefig('figE3_calibration.png', dpi=140, bbox_inches='tight')
plt.show()